In [1]:
import re
from pathlib import Path

def parse_morphological_features(features_str):
    """
    Parse morphological features from CoNLL-U format into unimorph-like tags.
    Example: 'Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin' -> 'V;IND;3;SG;PRS'
    """
    if not features_str or features_str == '_':
        return 'V'
    
    tags = ['V']  # Start with POS tag
    features = features_str.split('|')
    
    # Mapping from CoNLL-U features to unimorph abbreviations
    mappings = {
        'Mood': {
            'Ind': 'IND',
            'Sub': 'SBJV',
            'Imp': 'IMP',
            'Cnd': 'COND'
        },
        'Tense': {
            'Pres': 'PRS',
            'Past': 'PST',
            'Imp': 'IPFV',
            'Fut': 'FUT',
            'Pqp': 'PQP'
        },
        'Person': {
            '1': '1',
            '2': '2',
            '3': '3'
        },
        'Number': {
            'Sing': 'SG',
            'Plur': 'PL'
        },
        'VerbForm': {
            'Fin': None,  # Already captured by other features
            'Inf': 'INF',
            'Part': 'PTCP',
            'Ger': 'GER'
        }
    }
    
    for feature in features:
        if '=' not in feature:
            continue
        key, value = feature.split('=', 1)
        if key in mappings and value in mappings[key]:
            mapped = mappings[key][value]
            if mapped:  # Only add if not None
                tags.append(mapped)
    
    return ';'.join(tags)


def process_conllu_sentence(lines):
    """
    Process a single CoNLL-U sentence block.
    Returns tuple: (lemma, tags, form, context) or None if no verb found.
    """
    context = None
    verb_data = None
    
    for line in lines:
        line = line.strip()
        
        # Extract context from text comment
        if line.startswith('# text = '):
            context = line[9:].strip()
            continue
        
        # Skip comments and empty lines
        if line.startswith('#') or not line:
            continue
        
        # Parse token line
        parts = line.split('\t')
        if len(parts) < 6:
            continue
        
        token_id = parts[0]
        # Skip multiword tokens (e.g., "1-2")
        if '-' in token_id or '.' in token_id:
            continue
        
        form = parts[1]        # Inflected form
        lemma = parts[2]       # Lemma (infinitive)
        upos = parts[3]        # Universal POS tag
        feats = parts[5]       # Morphological features
        
        # Check if this is a verb (first one we encounter)
        if upos == 'VERB' and verb_data is None:
            tags = parse_morphological_features(feats)
            verb_data = (lemma, tags, form)
    
    # Return the verb data with context if both were found
    if verb_data and context:
        return verb_data + (context,)
    return None


def convert_conllu_to_unimorph(input_file, output_file):
    """
    Convert a CoNLL-U file to unimorph format with context.
    """
    sentence_lines = []
    results = []
    
    with open(input_file, 'r', encoding='utf-8') as fin:
        for line in fin:
            line = line.rstrip('\n')
            
            # Empty line marks end of sentence
            if not line:
                if sentence_lines:
                    result = process_conllu_sentence(sentence_lines)
                    if result:
                        results.append(result)
                    sentence_lines = []
            else:
                sentence_lines.append(line)
        
        # Process last sentence if file doesn't end with empty line
        if sentence_lines:
            result = process_conllu_sentence(sentence_lines)
            if result:
                results.append(result)
    
    # Write output
    with open(output_file, 'w', encoding='utf-8') as fout:
        for lemma, tags, form, context in results:
            fout.write(f"{lemma}\t{tags}\t{form}\t{context}\n")
    
    return len(results)


print("CoNLL-U to Unimorph converter ready!")

CoNLL-U to Unimorph converter ready!


## Test with a sample

In [2]:
# Test with sample data
sample_conllu = """# sent_id = 1
# text = Aquele cliente gosta apenas de vinho branco .
1	Aquele	aquele	DET	DEM	Gender=Masc|Number=Sing|PronType=Dem	2	det	_	_
2	cliente	cliente	NOUN	CN	Gender=Masc|Number=Sing	3	nsubj	_	_
3	gosta	gostar	VERB	V	Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin	0	root	_	_
4	apenas	apenas	ADV	ADV	_	6	advmod	_	_
5	de	de	ADP	PREP	_	6	case	_	_
6	vinho	vinho	NOUN	CN	Gender=Masc|Number=Sing	3	obl	_	_
7	branco	branco	ADJ	ADJ	Gender=Masc|Number=Sing	6	amod	_	_
8	.	.	PUNCT	PNT	_	3	punct	_	_

"""

# Write sample to file
with open('sample.conllu', 'w', encoding='utf-8') as f:
    f.write(sample_conllu)

# Process it
lines = sample_conllu.strip().split('\n')
result = process_conllu_sentence(lines)

if result:
    lemma, tags, form, context = result
    print("Sample output:")
    print(f"{lemma}\t{tags}\t{form}\t{context}")
else:
    print("No verb found in sample")

Sample output:
gostar	V;IND;SG;3;PRS	gosta	Aquele cliente gosta apenas de vinho branco .


## Convert Portuguese CoNLL-U file

In [ ]:
# Convert the Portuguese training file
input_file = 'pt_cintil-ud-train.conllu'
output_file = 'pt_cintil_verbs_with_context.txt'

if Path(input_file).exists():
    count = convert_conllu_to_unimorph(input_file, output_file)
    print(f"Processed {count} sentences with verbs")
    print(f"Output written to: {output_file}")
    
    # Show first few lines
    print("\nFirst 5 examples:")
    with open(output_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            parts = line.strip().split('\t')
            if len(parts) == 4:
                print(f"{i+1}. {parts[0]} [{parts[1]}] -> {parts[2]}")
                print(f"   Context: {parts[3][:80]}..." if len(parts[3]) > 80 else f"   Context: {parts[3]}")
else:
    print(f"File not found: {input_file}")
    print("Please make sure the file is in the current directory.")

## Process all Portuguese CoNLL-U files

In [3]:
# Process train, dev, and test files if they exist
files_to_process = [
    ('pt_cintil-ud-train.conllu', 'pt_verbs_context.trn'),
    ('pt_cintil-ud-dev.conllu', 'pt_verbs_context.dev'),
    ('pt_cintil-ud-test.conllu', 'pt_verbs_context.tst')
]

for input_file, output_file in files_to_process:
    if Path(input_file).exists():
        count = convert_conllu_to_unimorph(input_file, output_file)
        print(f"✓ {input_file}: {count} sentences -> {output_file}")
    else:
        print(f"✗ {input_file}: not found")

✓ pt_cintil-ud-train.conllu: 20674 sentences -> pt_verbs_context.trn
✓ pt_cintil-ud-dev.conllu: 2869 sentences -> pt_verbs_context.dev
✓ pt_cintil-ud-test.conllu: 3163 sentences -> pt_verbs_context.tst
✓ pt_cintil-ud-test.conllu: 3163 sentences -> pt_verbs_context.tst
